# Day 042 — Exercise 5: join_summary

**What you'll build:** `join_summary(conn) -> list[dict]` — JOIN the `orders` and `products` tables on the product name, then aggregate by region and category.

**Why it matters:** JOINs are the defining feature of relational databases. The `products` table holds category and unit_price — metadata that belongs to the product, not to each order. By JOINing, you enrich each order row with product metadata without duplicating it in the orders table. `INNER JOIN ... ON o.product = p.product` matches rows where the product name appears in both tables.

## Provided: All Helper Functions

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sqlite3
import pandas as pd


import sqlite3

def setup_db(conn):
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS orders (
            order_id  INTEGER PRIMARY KEY,
            product   TEXT,
            category  TEXT,
            region    TEXT,
            price     REAL,
            quantity  INTEGER,
            revenue   REAL
        )''')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS products (
            product    TEXT PRIMARY KEY,
            category   TEXT,
            unit_price REAL
        )''')
    rows = [
        (1,'Widget','Electronics','North',25.0,10,250.0),
        (2,'Gadget','Electronics','South',150.0,3,450.0),
        (3,'Widget','Electronics','South',25.0,5,125.0),
        (4,'Doohickey','Accessories','East',8.0,50,400.0),
        (5,'Gadget','Electronics','East',150.0,7,1050.0),
        (6,'Widget','Electronics','East',25.0,4,100.0),
        (7,'Doohickey','Accessories','North',8.0,20,160.0),
        (8,'Gadget','Electronics','North',150.0,2,300.0),
        (9,'Widget','Electronics','West',25.0,6,150.0),
        (10,'Doohickey','Accessories','South',8.0,15,120.0),
        (11,'Thingamajig','Accessories','North',200.0,1,200.0),
        (12,'Thingamajig','Accessories','East',200.0,4,800.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?,?)', rows
    )
    products = [
        ('Widget','Electronics',25.0),
        ('Gadget','Electronics',150.0),
        ('Doohickey','Accessories',8.0),
        ('Thingamajig','Accessories',200.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO products VALUES (?,?,?)', products
    )
    conn.commit()


def run_query(conn, sql, params=()):
    cur = conn.cursor()
    cur.execute(sql, params)
    cols = [col[0] for col in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]


def filter_orders(conn, region=None, category=None, min_revenue=None):
    conditions = []
    params = []
    if region is not None:
        conditions.append('region = ?')
        params.append(region)
    if category is not None:
        conditions.append('category = ?')
        params.append(category)
    if min_revenue is not None:
        conditions.append('revenue >= ?')
        params.append(min_revenue)
    where = ('WHERE ' + ' AND '.join(conditions)) if conditions else ''
    sql = f'SELECT * FROM orders {where} ORDER BY order_id'
    return run_query(conn, sql, tuple(params))


def group_revenue(conn, group_col):
    sql = (
        f'SELECT {group_col}, SUM(revenue) AS total, '
        'COUNT(*) AS orders, ROUND(AVG(revenue), 2) AS avg_revenue '
        f'FROM orders GROUP BY {group_col} ORDER BY total DESC'
    )
    return run_query(conn, sql)

In [ ]:
conn = sqlite3.connect(':memory:')
setup_db(conn)

## Your Implementation

In [ ]:
def join_summary(conn):
    """
    JOIN orders and products, aggregate by region and category.

    SQL shape:
      SELECT o.region, p.category,
             SUM(o.revenue) AS total_revenue,
             COUNT(*) AS order_count
      FROM orders o
      INNER JOIN products p ON o.product = p.product
      GROUP BY o.region, p.category
      ORDER BY total_revenue DESC

    Returns:
        list[dict] — one dict per region/category combo
    """
    # TODO: build the SQL string (use run_query to execute)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'join_summary' in globals()
        passed += 1; print('\u2705 Check 1: join_summary is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a list of dicts
    try:
        result = join_summary(conn)
        assert isinstance(result, list) and len(result) > 0
        assert isinstance(result[0], dict)
        passed += 1; print('\u2705 Check 2: returns list of dicts')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: has region, category, total_revenue, order_count columns
    try:
        expected = {'region', 'category', 'total_revenue', 'order_count'}
        actual = set(result[0].keys())
        assert actual == expected, f'expected {expected}, got {actual}'
        passed += 1; print(f'\u2705 Check 3: correct columns {expected}')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: has 7 rows (4 regions x 2 categories, minus West+Accessories)
    try:
        assert len(result) == 7, \
            f'expected 7 region/category combos, got {len(result)}'
        passed += 1; print('\u2705 Check 4: 7 region/category combinations')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: first row has highest total_revenue (East Accessories = 1200.0)
    try:
        top = result[0]
        assert abs(top['total_revenue'] - 1200.0) < 0.01, \
            f'expected top total_revenue = 1200.0, got {top["total_revenue"]}'
        passed += 1; print(f'\u2705 Check 5: top row total_revenue = {top["total_revenue"]}')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def join_summary(conn):
    sql = (
        'SELECT o.region, p.category, '
        'SUM(o.revenue) AS total_revenue, COUNT(*) AS order_count '
        'FROM orders o '
        'INNER JOIN products p ON o.product = p.product '
        'GROUP BY o.region, p.category '
        'ORDER BY total_revenue DESC'
    )
    return run_query(conn, sql)
```

</details>